# **Projectwork 1: Exploring Deep Learning Techniques for Basic Point Cloud Analysis in Environmental Monitoring**

**Specialication:** Digital Environment  
**Authors:** Florian Klaver, Nicola Noger  
**Supervisor:** Martin Schüle  
**Submission Deadline:** 22.05.2025


---
## **Introduction**

**Here we shortly describe the project. Not too much detail rest in report**

### **Features**
| Field               | Description |
|---------------------|------------|
| **X**              | X coordinate (East-West) in projected coordinate system. |
| **Y**              | Y coordinate (North-South) in projected coordinate system. |
| **Z**              | Elevation (height) above a reference surface. |
| **intensity**      | Strength of the laser return signal. |
| **return_number**  | Order of the return (e.g., first, second, etc.). |
| **number_of_returns** | Total number of returns for a given laser pulse. |
| **scan_direction_flag** | Indicates scanning direction (0 = forward, 1 = backward). |
| **edge_of_flight_line** | Marks if a point is at the edge of a flight path. |
| **synthetic**      | Flag indicating if the point was generated artificially. |
| **key_point**      | Flag for important points (e.g., survey control points). |
| **withheld**       | Flag for points that should be ignored (e.g., noise). |
| **scan_angle_rank** | Angle of the laser scan (-90° to +90°). |
| **user_data**      | Custom field for additional information. |
| **point_source_id** | Identifier for the source of the point (e.g., flight line). |
| **gps_time**       | Timestamp of when the point was captured. |
| **classification** | Category of the point. <br> <br> **1**&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;= Unclassified  <br> **2**&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;= Ground  <br> **3**&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;= Vegetation  <br> **6**&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;= Building  <br> **9**&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;= Water  <br>**17**&nbsp;&nbsp;&nbsp;= Bridge  |




---
## **Setup**

In [1]:
# All imports

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import laspy
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
from scipy.spatial import cKDTree
from torch.utils.data import DataLoader, TensorDataset

# Enable vectorized graphics
%config InlineBackend.figure_formats = ["svg"]

---
## **Preprocessing**

#### Loading the data

In [2]:
path_flo = "C:/Users/flori/Desktop/pa-1_data/zurich_1.las"
# path_nicola = 

las = laspy.read(path_flo)

# List all available attributes in the LAS file
print("Available attributes in LAS file:")
for spec in las.point_format.dimension_names:
    print("-", spec)


# total number of points
n_points = len(las.points)
print("\nTotal number of points:", n_points)

print(las.header)
print(las.points)

Available attributes in LAS file:
- X
- Y
- Z
- intensity
- return_number
- number_of_returns
- scan_direction_flag
- edge_of_flight_line
- classification
- synthetic
- key_point
- withheld
- scan_angle_rank
- user_data
- point_source_id
- gps_time

Total number of points: 12902017
<LasHeader(1.2, <PointFormat(1, 0 bytes of extra dims)>)>
<ScaleAwarePointRecord(fmt: <PointFormat(1, 0 bytes of extra dims)>, len: 12902017, point size: 28)>


In [3]:
# Inforamtion for transforming the data into a pandas dataframe
print(type(las.x)) 
print(type(las.x[0]))  

<class 'laspy.point.dims.ScaledArrayView'>
<class 'numpy.float64'>


In [ ]:
# Create a pandas DataFrame with the LAS data for initial data exploration
df = pd.DataFrame({
    "X": np.array(las.x), 
    "Y": np.array(las.y),
    "Z": np.array(las.z),
    "intensity": np.array(las.intensity),
    "return_number": np.array(las.return_number),
    "number_of_returns": np.array(las.number_of_returns),
    "scan_direction_flag": np.array(las.scan_direction_flag),
    "edge_of_flight_line": np.array(las.edge_of_flight_line),
    "synthetic": np.array(las.synthetic),
    "key_point": np.array(las.key_point),
    "withheld": np.array(las.withheld),
    "user_data": np.array(las.user_data),
    "point_source_id": np.array(las.point_source_id),
    "scan_angle_rank": np.array(las.scan_angle_rank),
    "gps_time": np.array(las.gps_time),
    "classification": np.array(las.classification)
})


# get a first impression of the data
print(df.shape)
display(df.info())
display(df.describe())
display(df.head())
print("\nClassification values:")
display(df["classification"].value_counts())


(12902017, 16)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12902017 entries, 0 to 12902016
Data columns (total 16 columns):
 #   Column               Dtype  
---  ------               -----  
 0   X                    float64
 1   Y                    float64
 2   Z                    float64
 3   intensity            uint16 
 4   return_number        uint8  
 5   number_of_returns    uint8  
 6   scan_direction_flag  uint8  
 7   edge_of_flight_line  uint8  
 8   synthetic            uint8  
 9   key_point            uint8  
 10  withheld             uint8  
 11  user_data            uint8  
 12  point_source_id      uint16 
 13  scan_angle_rank      int8   
 14  gps_time             float64
 15  classification       uint8  
dtypes: float64(4), int8(1), uint16(2), uint8(9)
memory usage: 566.0 MB


None

,X,Y,Z,intensity,return_number,number_of_returns,scan_direction_flag,edge_of_flight_line,synthetic,key_point,withheld,user_data,point_source_id,scan_angle_rank,gps_time,classification
count,1.290202e+07,1.290202e+07,1.290202e+07,1.290202e+07,1.290202e+07,1.290202e+07,12902017.0,12902017.0,12902017.0,12902017.0,12902017.0,1.290202e+07,1.290202e+07,1.290202e+07,1.290202e+07,1.290202e+07
mean,2.683532e+06,1.247508e+06,4.311112e+02,1.346394e+02,1.374864e+00,1.750828e+00,0.0,0.0,0.0,0.0,0.0,5.225372e+01,1.226129e+02,-8.729442e-01,2.051168e+08,3.921719e+00
std,2.977153e+02,2.757434e+02,1.662515e+01,2.602035e+02,7.983605e-01,1.181962e+00,0.0,0.0,0.0,0.0,0.0,8.472675e+00,9.829875e-01,1.678507e+01,6.876755e+02,2.147887e+00
min,2.683000e+06,1.247000e+06,3.891200e+02,5.000000e+00,1.000000e+00,1.000000e+00,0.0,0.0,0.0,0.0,0.0,1.700000e+01,1.210000e+02,-3.000000e+01,2.051151e+08,1.000000e+00
25%,2.683251e+06,1.247295e+06,4.194700e+02,5.200000e+01,1.000000e+00,1.000000e+00,0.0,0.0,0.0,0.0,0.0,4.800000e+01,1.220000e+02,-1.400000e+01,2.051165e+08,2.000000e+00
50%,2.683573e+06,1.247499e+06,4.291200e+02,1.110000e+02,1.000000e+00,1.000000e+00,0.0,0.0,0.0,0.0,0.0,5.000000e+01,1.230000e+02,-3.000000e+00,2.051165e+08,3.000000e+00
75%,2.683791e+06,1.247744e+06,4.396300e+02,1.770000e+02,1.000000e+00,2.000000e+00,0.0,0.0,0.0,0.0,0.0,5.300000e+01,1.230000e+02,1.300000e+01,2.051173e+08,6.000000e+00
max,2.684000e+06,1.248000e+06,5.158300e+02,6.553400e+04,7.000000e+00,7.000000e+00,0.0,0.0,0.0,0.0,0.0,2.400000e+02,1.250000e+02,3.000000e+01,2.051179e+08,1.700000e+01


,X,Y,Z,intensity,return_number,number_of_returns,scan_direction_flag,edge_of_flight_line,synthetic,key_point,withheld,user_data,point_source_id,scan_angle_rank,gps_time,classification
0,2683033.91,1247000.0,433.53,71,1,1,0,0,0,0,0,47,125,29,2.051151e+08,6
1,2683074.91,1247000.0,426.62,73,1,2,0,0,0,0,0,46,125,29,2.051151e+08,6
2,2683086.29,1247000.0,433.13,114,1,1,0,0,0,0,0,49,125,29,2.051151e+08,6
3,2683135.96,1247000.0,417.98,106,2,2,0,0,0,0,0,67,125,28,2.051151e+08,6
4,2683139.50,1247000.0,431.08,183,1,1,0,0,0,0,0,50,125,28,2.051151e+08,6



Classification values:


classification
6     5217866
2     4469036
3     2829680
1      291256
17      90614
9        3565
Name: count, dtype: int64

In [10]:
df_vegetation = df[df["classification"] == 3]

print(df_vegetation.head())

              X           Y       Z  intensity  return_number  \
5    2683161.49  1247000.00  427.91         11              2   
7    2683164.57  1247000.00  423.75         10              2   
165  2683533.93  1247006.97  408.09         52              1   
179  2683533.51  1247007.65  408.38         38              1   
180  2683533.46  1247007.16  408.43         46              1   

     number_of_returns  scan_direction_flag  edge_of_flight_line  synthetic  \
5                    3                    0                    0          0   
7                    3                    0                    0          0   
165                  1                    0                    0          0   
179                  1                    0                    0          0   
180                  1                    0                    0          0   

     key_point  withheld  user_data  point_source_id  scan_angle_rank  \
5            0         0         68              125         

---
## **Explorative Data Analysis**

---
## **Feature Engineering**

---
## **Model Developement**

---
## **Model Deployment**